In [1]:
import os
import numpy as np
import pandas as pd
import random
import pickle
import torch
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO

import shap

In [2]:
# Set device and seed
os.environ["CUDA_VISIBLE_DEVICES"] = "6"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

seed = 316
random.seed(seed)
np.random.seed(seed)

In [3]:
with open('../data/train_sequences_idx.pkl', 'rb') as f:
    train_sequences_idx = pickle.load(f)
with open('../data/test_sequences_idx.pkl', 'rb') as f:
    test_sequences_idx = pickle.load(f)

with open('../data/idx2user.pkl', 'rb') as f:
    idx2user = pickle.load(f)
with open('../data/idx2movie.pkl', 'rb') as f:
    idx2movie = pickle.load(f)
    
with open('../data/ratings_dict.pkl', 'rb') as f:
    ratings_dict = pickle.load(f)

user_embeddings = np.load('../data/user_embeddings.npy')
movie_embeddings = np.load('../data/movie_embeddings.npy')

with open('../data/movie_index_to_title.pkl', 'rb') as f:
    movie_index_to_title = pickle.load(f)

# Dictionary that maps MovieID to Title for Quick Lookup
with open('../data/movie_id_to_title.pkl', 'rb') as f:
    movie_id_to_title = pickle.load(f)

In [4]:
# Logd Model
version_number = 3
total_timesteps = 500_000
model = PPO.load(f"../models/PPO_Ver_{version_number}_{total_timesteps}")

/home/stu5/s5/law3082/miniconda3/envs/idai610/lib/python3.10/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [5]:
# Load saved evaluation trajectories
with open('../results/PPO_session_trajectories.pkl', 'rb') as f:
    sessions = pickle.load(f)

# Env

In [6]:
# Create list of User Indices that you can legitimately evaluate
evaluable_users = []
for user_idx in train_sequences_idx.keys():
    # Count positive items in training
    pos_train = 0
    for movie_idx in train_sequences_idx[user_idx]:
        rating = ratings_dict.get((user_idx, movie_idx), 0)
        if rating >= 4:
            pos_train += 1

    # if # of positively rated movies in the train set < 5, skip the rest of the for loop
    if pos_train < 5:
        continue
    
    # Check test set has at least one item
    test_items = test_sequences_idx.get(user_idx, [])
    if len(test_items) == 0:
        continue
    
    # Check test set has at least one positive item
    has_positive_test = False
    for movie_idx in test_items:
        rating = ratings_dict.get((user_idx, movie_idx), 0)
        if rating >= 4:
            has_positive_test = True
            break
    
    if not has_positive_test:
        continue
    
    evaluable_users.append(user_idx)

print(f"Number of evaluable users: {len(evaluable_users)}")

Number of evaluable users: 5961


# SHAP

In [7]:
def get_state(user_idx, history_indices, user_embeddings, movie_embeddings, item_weights=None):
    u = user_embeddings[user_idx]
    item_embs = movie_embeddings[history_indices]
    if item_weights is not None:
        weights = np.exp(item_weights) / np.sum(np.exp(item_weights))
        weighted_embs = item_embs * weights[:, np.newaxis]
        g = np.sum(weighted_embs, axis=0)
    else:
        g = np.mean(item_embs, axis=0)
    u_g_interaction = u * g
    state = np.concatenate([u, u_g_interaction, g])
    return state.astype(np.float32)

In [8]:
def predict_from_history_mask(masks, user_idx, history_list, target_idx):
    scores = []
    for mask in masks:
        # Select movies where mask == 1 (or > 0.5 for continuous masks)
        selected = [history_list[i] for i in range(len(mask)) if mask[i] > 0.5]
        if len(selected) == 0:
            selected = [history_list[0]]   # fallback to first movie
        
        state = get_state(user_idx, selected, user_embeddings, movie_embeddings)
        # PPO expects observation as a numpy array; we can keep it on CPU
        action, _ = model.predict(state, deterministic=True)
        target_emb = movie_embeddings[target_idx]
        score = np.dot(action, target_emb)
        scores.append(score)
    return np.array(scores)

In [9]:
# Function to explain 1 step
def explain_saved_step(user_idx, step_data):
    history_at_time = step_data['history_at_time']   # list of movie indices
    target_idx = step_data['recommended_movie_idx']
    
    from functools import partial
    pred_func = partial(predict_from_history_mask,
                        user_idx=user_idx,
                        history_list=history_at_time,
                        target_idx=target_idx)
    
    # Background masks: random binary masks (e.g., 50 samples)
    background_masks = np.random.randint(0, 2, size=(50, len(history_at_time)))
    explainer = shap.KernelExplainer(pred_func, background_masks)
    # Explain the full history (all ones)
    shap_values = explainer.shap_values(np.ones(len(history_at_time)))
    return shap_values

In [10]:
def get_influence_label(weight):
    return "+" if weight > 0 else "-"

# SHAP Evaluation

In [11]:
def SHAP_eval(user_index, user_id, sessions, top_n_impactful=5):
    # user_index = 0
    rec_movies_explanations = []
    steps = sessions[user_index]['steps']

    # Print trajectory
    print(f"Recommendations for USER {user_id} & their Actual Rating:")
    ground_truth_df = []
    for s in steps:
        ground_truth_df.append({
            'Step': s['step'],
            'Movie Title': movie_index_to_title.get(s['recommended_movie_idx'], 'Unknown'),
            'Rating': s['rating']
        })
    ground_truth_df = pd.DataFrame(ground_truth_df)
    print(ground_truth_df.to_string(index=False))
    print("\n")
    
    for i, step in enumerate(steps):   
        # print(step)
        # {'step': 1, 'recommended_movie_idx': 0, 'recommended_movie_title': 'Toy Story (1995)', 'rating': 5.0, 'score': 3.5080097, 'reward': 1.0, 'history_at_time': [964, 580, 2205, 1421, 513]}    
        recommended_movie_title = step['recommended_movie_title']
        recommended_movie_user_rating = step['rating']
        print(f"Step {i+1}: Recommended {recommended_movie_title} (rating {recommended_movie_user_rating})")

        shap_vals = explain_saved_step(user_index, step)
        user_id = idx2user[user_index]
        # print(f"User ID {user_id} SHAP values: {shap_vals}")
        # User ID 1 SHAP values: [-0.03141301  0.00831559  0.02534873 -0.05784124  0.10574878]
        
        history_at_time = step['history_at_time']
        
        history_titles = []
        for idx in history_at_time:
            history_titles.append(movie_id_to_title[idx2movie[idx]])
        
        title_to_rating = {}
        for movie_index in history_at_time:
            # Get the title given movie_index
            title = movie_id_to_title[idx2movie[movie_index]]
            # Get the user's rating for this movie; 'N/A' if not found
            rating = ratings_dict.get((user_index, movie_index), 'N/A')
            # Save in dictionary
            title_to_rating[title] = rating
        
        history_influence = []
        for i in range(len(history_at_time)):
            movie_title = history_titles[i]
            shap_weight = shap_vals[i]
            history_influence.append((movie_title, shap_weight))
        
        most_impactful_movies = pd.DataFrame(history_influence, columns=["Movie in History", "Weight"])
        most_impactful_movies["Rating"] = most_impactful_movies["Movie in History"].map(title_to_rating)    
        most_impactful_movies["Influence"] = most_impactful_movies["Weight"].apply(get_influence_label)
        most_impactful_movies["Weight"] = most_impactful_movies["Weight"].round(2)
        most_impactful_movies = most_impactful_movies.sort_values("Weight", ascending=False).head(top_n_impactful)
        most_impactful_movies = most_impactful_movies[["Movie in History", "Rating", "Weight", "Influence"]]

        # print(most_impactful_movies.to_string())
        print(most_impactful_movies)
        print("\n")
        rec_movies_explanations.append(most_impactful_movies)

    return rec_movies_explanations

In [12]:
users_to_explain = [1, 3, 40, 67, 316]
# users_to_explain = [1, 316]

all_explanations = {}
for user_index in evaluable_users:
    user_id = idx2user[user_index]
    if user_id in users_to_explain:
        # print(f"User ID: {user_id}")
        # Get a list of 10 explanations
        # Each explanation is the top 5 movies why 1 movie was recommended to the user
        rec_movies_explanations = SHAP_eval(user_index, user_id, sessions)

        # Save this list of explanations
        all_explanations[user_index] = rec_movies_explanations
        print("\n")

# This is a Dictionary
# Keys = User Index
# Values = List of 10 explanations
# There are 5 users
# Each user was recommeded 10 movies
# Each recommended movie has 1 explanation
# Each explanation consists of 5 historical rated movies that were most impactful in decided to recommend it to the user
# SO: 5 * 10 * 5 = 250 explanations
with open(f'PPO_SHAP_explanations.pkl', 'wb') as f:
    pickle.dump(all_explanations, f)

Recommendations for USER 1 & their Actual Rating:
 Step                         Movie Title  Rating
    1                    Toy Story (1995)     5.0
    2                Bug's Life, A (1998)     5.0
    3         Beauty and the Beast (1991)     5.0
    4                        Mulan (1998)     4.0
    5                      Aladdin (1992)     4.0
    6               Close Shave, A (1995)     3.0
    7                         Antz (1998)     4.0
    8                       Tarzan (1999)     3.0
    9 Hunchback of Notre Dame, The (1996)     4.0
   10                   Pocahontas (1995)     5.0


Step 1: Recommended Toy Story (1995) (rating 5.0)
                         Movie in History  Rating  Weight Influence
4                 Schindler's List (1993)     5.0    0.04         +
1  Snow White and the Seven Dwarfs (1937)     4.0    0.01         +
2           Miracle on 34th Street (1947)     4.0   -0.00         -
3                          Ponette (1996)     4.0   -0.00         -
0       